## 05.03节练习参考答案

### 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

# 本 notebook 位于 answers/ 子目录下，src 包在其上一级目录。
# 把上层目录加入 sys.path，才能 from src.pto_layers import ...
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math

from src.pto_layers import PyPTOLinear, PyPTOReLU, PyPTOLazyLinear

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch05/ch05) ，并在此基础上补充了 PyPTO 的实现。  

### 练习5.3.1
如果指定了第一层的输入尺寸，但没有指定后续层的尺寸，会发生什么？是否立即进行初始化？

**解答：**
可以正常运行。第一层会立即初始化,但其他层是直到数据第一次通过模型传递才会初始化。

In [3]:
"""延后初始化"""
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.LazyLinear(10))
# 尚未初始化
print(net)

X = torch.rand(2, 20)
net(X)
print(net)

Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=10, bias=True)
)
Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)


参考的答案的写法为：

In [ ]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))

第一层仍使用 `LazyLinear`，未指定输入尺寸，与题目要求（指定第一层输入尺寸）不符。

**PyPTO 版**

In [5]:
"""延后初始化"""
net = nn.Sequential(PyPTOLinear(20, 256), PyPTOReLU(), PyPTOLazyLinear(10))
# 尚未初始化
print(net)

X = torch.rand(2, 20).npu()
net(X)
print(net)

Sequential(
  (0): PyPTOLinear(in_features=20, out_features=256, bias=True)
  (1): PyPTOReLU()
  (2): PyPTOLazyLinear(in_features=0, out_features=10, bias=True)
)
Sequential(
  (0): PyPTOLinear(in_features=20, out_features=256, bias=True)
  (1): PyPTOReLU()
  (2): PyPTOLazyLinear(in_features=256, out_features=10, bias=True)
)


### 练习5.3.2
如果指定了不匹配的维度会发生什么？

**解答：**
会由于矩阵乘法的维度不匹配而报错。在下面的代码中便指定了不匹配的维度。由于第一层 nn.Linear(20, 256) 的输入维度为 20，所以输入数据 X 的最后一维必须为 20 才能与该层的权重矩阵相乘。

In [4]:
net = nn.Sequential(
    nn.Linear(20, 256), nn.ReLU(),
    nn.LazyLinear(128), nn.ReLU(),
    nn.LazyLinear(10))

X = torch.rand(2, 10)
try:
    net(X)
except Exception as e:
    print(e)

mat1 and mat2 shapes cannot be multiplied (2x10 and 20x256)


**PyPTO 版**

In [5]:
net = nn.Sequential(
    PyPTOLinear(20, 256), PyPTOReLU(),
    PyPTOLazyLinear(128), PyPTOReLU(),
    PyPTOLazyLinear(10))

X = torch.rand(2, 10).npu()
try:
    net(X)
except Exception as e:
    print(e)

期望输入维度 20, 得到 10


### 练习5.3.3
如果输入具有不同的维度，需要做什么？提示：查看参数绑定的相关内容。

**解答：**
输入维度改变时，入口层的输入特征数必须与新维度一致，因此需要更换为匹配新维度的入口层。注意：参数绑定要求两层形状完全一致，跨维度无法绑定，因此入口层只能整体替换，不能只换权重。

若希望在新旧网络间复用形状不变的后续层参数，可借助参数绑定（见 05.02 节）让多个层共享同一份权重。下方将后续层 `shared`（256→128）在输入维度为 20 和 10 的两个网络间共享。

In [6]:
# 原始网络（输入维度 20）
shared = nn.Linear(256, 128)            # 后续层，形状不随输入维度变化，可用于参数绑定
net = nn.Sequential(
    nn.Linear(20, 256), nn.ReLU(),
    shared, nn.ReLU(),
    nn.Linear(128, 10))

# 输入维度改为 10：入口层 in_features 不同，无法绑定，需整体替换；
# 后续层形状一致，通过参数绑定（共享 shared 这一层）在新旧网络间复用参数。
net2 = nn.Sequential(
    nn.Linear(10, 256), nn.ReLU(),
    shared, nn.ReLU(),
    nn.Linear(128, 10))

X = torch.rand(2, 10)
net2(X)

tensor([[-0.0893,  0.1470, -0.0264,  0.0241, -0.1638, -0.0401, -0.0017,  0.0201,
         -0.1050, -0.0204],
        [-0.0987,  0.0413,  0.0069,  0.0366, -0.1109,  0.0266, -0.0384,  0.0077,
         -0.0622, -0.0362]], grad_fn=<AddmmBackward0>)

参考的答案的写法为：  

添加一个额外的线性层，并将第一个线性层的权重与该层的权重绑定在一起。这样就可以解决维度不匹配的问题，并且保持模型的权重不变。注意，在上面的代码中，我们假设第一个线性层的偏置项为零，因此不需要对其进行参数绑定。

In [ ]:
net = nn.Sequential(
    nn.Linear(20, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10))

# 添加额外的线性层
extra_layer = nn.Linear(10, 256)

# 将第一个线性层与额外的线性层的权重进行绑定
net[0].weight = extra_layer.weight

该写法存在以下错误：

1. **“保持权重不变”不成立**：`net[0].weight` 由 `(256, 20)` 被替换为 `(256, 10)`，入口维度从 20 变为 10。
2. **不属于参数绑定**：绑定要求形状一致，跨维度无法绑定；此处仅更换入口层权重来源，未在层间共享参数。
3. **“偏置项为零”不成立**：`nn.Linear` 偏置由 `reset_parameters` 随机初始化，并非零。

该写法能运行仅因 `nn.Linear` 不校验 `in_features`。

**PyPTO 版**

In [8]:
# 原始网络（输入维度 20）
shared = PyPTOLinear(256, 128)          # 后续层，形状不随输入维度变化，可用于参数绑定
net = nn.Sequential(
    PyPTOLinear(20, 256), PyPTOReLU(),
    shared, PyPTOReLU(),
    PyPTOLinear(128, 10))

# 输入维度改为 10：PyPTOLinear 会校验 in_features，且 _kernel 按维度编译，
# 入口层不能只替换 weight，需整体替换；后续层通过参数绑定（共享 shared）复用。
net2 = nn.Sequential(
    PyPTOLinear(10, 256), PyPTOReLU(),
    shared, PyPTOReLU(),
    PyPTOLinear(128, 10))

X = torch.rand(2, 10).npu()
net2(X)

tensor([[ 0.0206, -0.0634, -0.0931,  0.0884,  0.0454,  0.0624,  0.0845, -0.1575,
          0.0881, -0.0632],
        [ 0.0585, -0.0522, -0.0731,  0.0653,  0.0653,  0.0577,  0.0714, -0.1154,
          0.0899, -0.0487]], device='npu:0', grad_fn=<ViewBackward0>)

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击展开 / 折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Linear</code>在前向计算时不校验<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">in_features</code>，只按权重张量的实际形状计算，因此直接替换权重即可运行；而<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">PyPTOLinear.forward</code>会校验<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">in_features</code>，且其<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">_kernel</code>在构造时按维度编译，因此必须整体替换该层，而不能只替换权重。</li>
    </ul>
  </div>
</details>